# BondSpot dashboard — duration & debt composition over time

Wykresy:
1. **Portfolio-weighted metryki** (Mod/Mac Duration, ATM, ATR) w czasie
2. **Per typ obligacji** (DS/PS/WS/WZ/IZ/...) — facet 2×2
3. **Skład długu** — stacked area (bondy + bony skarbowe)

## Setup
1. `pip install -r ../requirements-notebook.txt`
2. Utwórz `.env` w roocie repo z `SUPABASE_URL` i `SUPABASE_SERVICE_ROLE_KEY`
3. Run all cells

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import requests
from dotenv import load_dotenv
from plotly.subplots import make_subplots

# Renderer wymagany zeby fig.show() emitowal HTML/JS przy nbconvert -> HTML;
# live Jupyter zwykle uzywa 'plotly_mimetype' ale to nie renderuje w GH Pages.
pio.renderers.default = "notebook_connected"

load_dotenv(Path("..") / ".env")

SUPABASE_URL = os.environ["SUPABASE_URL"].rstrip("/")
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

HEADERS = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
}

PAGE_SIZE = 1000  # Supabase PGRST_DB_MAX_ROWS default - kapuje response do 1000

# Wszystkie statystyki i wykresy startuja od 2012-01-01 (tylko aukcje
# holenderskie/Dutch uniform-price). Polski MF przeszedl na Dutch ~2010,
# ale 2012 jako clean cutoff.
START_DATE = pd.Timestamp("2012-01-01")
START_DATE_STR = START_DATE.strftime("%Y-%m-%d")


def _paginate(method, url, *, json_body=None, timeout=120, max_pages=200):
    """Stronicowane wywolanie endpoint-u Supabase. PostgREST tnie response do
    PGRST_DB_MAX_ROWS (1000 default na Supabase) niezaleznie od Range header,
    wiec idziemy w petli inkrementujac offset i konkatenujac wyniki."""
    rows: list = []
    for page in range(max_pages):
        offset = page * PAGE_SIZE
        h = {**HEADERS,
             "Range-Unit": "items",
             "Range": f"{offset}-{offset + PAGE_SIZE - 1}"}
        kw = {"headers": h, "timeout": timeout}
        if json_body is not None:
            kw["json"] = json_body
        r = requests.request(method, url, **kw)
        if r.status_code not in (200, 206):
            r.raise_for_status()
        chunk = r.json()
        if not isinstance(chunk, list):
            return chunk  # scalar / dict response - return as-is
        rows.extend(chunk)
        if len(chunk) < PAGE_SIZE:
            return rows
    raise RuntimeError(f"_paginate hit max_pages={max_pages} without finishing")


def rpc(name: str, payload: dict | None = None) -> pd.DataFrame:
    rows = _paginate("POST", f"{SUPABASE_URL}/rest/v1/rpc/{name}",
                     json_body=payload or {})
    return pd.DataFrame(rows)


def fetch_view(name: str, query: str = "?select=*") -> pd.DataFrame:
    rows = _paginate("GET", f"{SUPABASE_URL}/rest/v1/{name}{query}")
    return pd.DataFrame(rows)


print(f"Connected. START_DATE = {START_DATE_STR}")

## 1. Portfolio-weighted metryki w czasie

Z `v_portfolio_metrics_daily` (ważone outstanding-em dziennym, bondy hurtowe).

In [ ]:
df1 = fetch_view(
    "v_portfolio_metrics_daily",
    f"?fixing_date=gte.{START_DATE_STR}&select=*&order=fixing_date.asc",
)
df1["fixing_date"] = pd.to_datetime(df1["fixing_date"])
for c in ["portfolio_mod_duration", "portfolio_mac_duration", "portfolio_atm",
         "portfolio_atr", "portfolio_yield_pct", "total_outstanding_mln_pln"]:
    if c in df1.columns:
        df1[c] = pd.to_numeric(df1[c], errors="coerce")

print(f"Days: {len(df1)},  range: {df1.fixing_date.min().date()} → {df1.fixing_date.max().date()}")
df1.tail()

In [ ]:
fig = go.Figure()
for col, label in [
    ("portfolio_mod_duration", "Modified Duration"),
    ("portfolio_mac_duration", "Macaulay Duration"),
    ("portfolio_atm", "ATM (years to maturity)"),
    ("portfolio_atr", "ATR (years to refixing)"),
]:
    fig.add_trace(go.Scatter(x=df1["fixing_date"], y=df1[col], name=label, mode="lines"))

fig.update_layout(
    title="Portfolio-weighted metryki polskiego długu (bondy hurtowe)",
    xaxis_title="Data fixingu (EOD = sesja 2)",
    yaxis_title="Lata",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 2. Per rodzaj kuponu (I / OS / Z)

Każdy z 4 paneli pokazuje jedną metrykę ważoną outstanding-em, kolory = bucket kuponowy:
- **I** — inflacyjne (IZ)
- **OS** — stałe + zerokuponowe (OK, DS, PS, WS, …)
- **Z** — zmienne (WZ, NZ)

Czego się spodziewać:
- **OS** — duration zależne od mixu krótkich (OK) i długich (WS) — średnio rośnie z trendem wydłużania krzywej
- **Z** (floatery) — Mod/Mac ≈ ATR (≤ 0.5Y), bo resetują kupon co 6mc
- **I** (inflation-linked) — Mod/Mac niskie, ATM długie (długie wykupy ale małe duration bo CPI-link)

In [ ]:
# Bucket mapping wspolny dla chart 2 i 3b (definiujemy raz).
BOND_TYPE_TO_BUCKET = {
    "IZ": "I",
    "WZ": "Z", "NZ": "Z",
}

def to_bucket(bt: str) -> str:
    if bt == "tbill":
        return "tbill"
    return BOND_TYPE_TO_BUCKET.get(bt, "OS")

df2 = fetch_view(
    "v_portfolio_metrics_by_type",
    f"?fixing_date=gte.{START_DATE_STR}&select=*&order=fixing_date.asc,bond_type.asc",
)
df2["fixing_date"] = pd.to_datetime(df2["fixing_date"])
for c in ["total_mln_pln", "w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]:
    df2[c] = pd.to_numeric(df2[c], errors="coerce")

df2["bucket"] = df2["bond_type"].map(to_bucket)

# Re-aggregacja per (fixing_date, bucket): weighted_avg po outstanding.
# Kazda metryka w v_portfolio_metrics_by_type to juz weighted_avg per typ
# (sum(metric_isin * outstanding_isin) / sum(outstanding_isin)). Zeby
# polaczyc kilka typow w bucket bierzemy:
#   bucket_w_metric = sum(w_metric_t * total_t) / sum(total_t)
# co matematycznie odpowiada laczeniu surowych (metric_isin * outstanding_isin)
# z wszystkich typow w buckecie.
METRICS = ["w_mod_duration", "w_mac_duration", "w_atm", "w_atr", "w_yield_pct"]
for m in METRICS:
    # NaN-safe: jesli w_metric jest NaN (np. w_yield_pct dla pewnych dni) to
    # ten kawalek nie wchodzi ani do licznika ani do mianownika.
    df2[f"_num_{m}"] = df2[m] * df2["total_mln_pln"]
    df2[f"_den_{m}"] = df2["total_mln_pln"].where(df2[m].notna())

agg = df2.groupby(["fixing_date", "bucket"], as_index=False).agg(
    total_mln_pln=("total_mln_pln", "sum"),
    **{f"_num_{m}": (f"_num_{m}", "sum") for m in METRICS},
    **{f"_den_{m}": (f"_den_{m}", "sum") for m in METRICS},
)
for m in METRICS:
    agg[m] = agg[f"_num_{m}"] / agg[f"_den_{m}"].replace(0, pd.NA)
    agg.drop(columns=[f"_num_{m}", f"_den_{m}"], inplace=True)

df2 = agg  # od teraz df2 ma kolumny: fixing_date, bucket, total_mln_pln, w_*

print(f"Rows: {len(df2)},  buckets: {sorted(df2.bucket.unique())},  "
      f"range: {df2.fixing_date.min().date()} → {df2.fixing_date.max().date()}")
df2.tail()

In [ ]:
metrics = [
    ("w_mod_duration", "Modified Duration (lata)"),
    ("w_mac_duration", "Macaulay Duration (lata)"),
    ("w_atm", "ATM (lata)"),
    ("w_atr", "ATR (lata)"),
]
fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

# Spojny color scheme z chart 3b
BUCKET_COLORS_2 = {
    "OS": "#1f77b4",  # niebieski
    "Z": "#ff7f0e",   # pomaranczowy
    "I": "#9467bd",   # fioletowy
}
buckets_sorted = [b for b in ["OS", "Z", "I"] if b in df2.bucket.unique()]

for i, (col, _) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    for bucket in buckets_sorted:
        sub = df2[df2.bucket == bucket].sort_values("fixing_date")
        fig.add_trace(
            go.Scatter(
                x=sub["fixing_date"], y=sub[col],
                name=bucket, legendgroup=bucket,
                showlegend=(i == 0), mode="lines",
                line=dict(color=BUCKET_COLORS_2[bucket], width=1.8),
            ),
            row=row, col=c,
        )

fig.update_layout(
    title="Metryki ważone outstanding per rodzaj kuponu (I / OS / Z)",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.05),
)
fig.show()

## 3. Skład długu — stacked area

Bondy hurtowe per typ + bony skarbowe (jako jeden kubełek `tbill`). Pokazuje jak ewoluowała struktura zadłużenia.

Resample do miesięcznych snapshotów (end-of-month), inaczej wykres jest zaszumiony przy 3000+ dni × 8 typów.

In [ ]:
TODAY = pd.Timestamp.today().normalize()

# Event-driven sklad dlugu z MF (niezalezne od BondSpota - chwytamy NZ-tki
# od dnia pierwszej emisji w MF, nie od pierwszego BondSpotowego fixingu).
# Widoki v_bond_outstanding_by_type_events i v_tbill_outstanding_events
# robia cumulative sum delty per typ na poziomie SQL.

bond_events = fetch_view(
    "v_bond_outstanding_by_type_events",
    "?select=*&order=change_date.asc,bond_type.asc",
)
bond_events["change_date"] = pd.to_datetime(bond_events["change_date"])
bond_events["outstanding_mln_pln"] = pd.to_numeric(
    bond_events["outstanding_mln_pln"], errors="coerce"
)

tbill_events = fetch_view(
    "v_tbill_outstanding_events",
    "?select=*&order=change_date.asc",
)
tbill_events["change_date"] = pd.to_datetime(tbill_events["change_date"])
tbill_events["outstanding_mln_pln"] = pd.to_numeric(
    tbill_events["outstanding_mln_pln"], errors="coerce"
)

# Diagnostyka source: dla kazdego typu pokazujemy ile eventow i ostatnia
# wartosc cumulative (z SQL). Latest_at_today odzwierciedla wartosc po
# odfiltrowaniu przyszlych synthetic redemption events.
print("Source events per bond_type (with cumulative latest <= TODAY):")
diag = bond_events.copy()
diag_today = diag[diag["change_date"] <= TODAY]
src_stats = diag_today.groupby("bond_type").agg(
    n_events=("change_date", "count"),
    first_date=("change_date", "min"),
    last_date=("change_date", "max"),
    last_outstanding=("outstanding_mln_pln", "last"),
).sort_values("last_outstanding", ascending=False)
print(src_stats)
print(f"\ntbill source: {len(tbill_events)} events, "
      f"last <= TODAY: {tbill_events[tbill_events['change_date'] <= TODAY]['outstanding_mln_pln'].iloc[-1]:.1f} "
      f"on {tbill_events[tbill_events['change_date'] <= TODAY]['change_date'].max().date()}")

# Pelny dzienny index od najwczesniejszego eventu do TODAY.
# Reindex+ffill PER typ jest bardziej deterministyczne niz pivot+resample
# (pivot_table moze dla niektorych dtype-ow wstawic 0 zamiast NaN i wtedy
# ffill nie propaguje wartosci - co zlobalismy wczesniej).
data_min = min(bond_events["change_date"].min(), tbill_events["change_date"].min())
all_dates = pd.date_range(start=data_min, end=TODAY, freq="D")

wide = pd.DataFrame(index=all_dates)
for bt in sorted(bond_events["bond_type"].dropna().unique()):
    sub = bond_events[bond_events["bond_type"] == bt].sort_values("change_date")
    s = sub.set_index("change_date")["outstanding_mln_pln"]
    s = s[~s.index.duplicated(keep="last")]
    s = s[s.index <= TODAY]  # bez przyszlych redemption events
    wide[bt] = s.reindex(all_dates).ffill().fillna(0)

ts = tbill_events.set_index("change_date")["outstanding_mln_pln"]
ts = ts[~ts.index.duplicated(keep="last")]
ts = ts[ts.index <= TODAY]
wide["tbill"] = ts.reindex(all_dates).ffill().fillna(0)

# Trim do START_DATE (global z setup-a)
wide = wide[wide.index >= START_DATE]

# Reduce do dni z faktycznymi zmianami (auction-event datapointy zamiast daily)
mask = (wide != wide.shift(1)).any(axis=1)
wide = wide[mask]

# Order: typy od najwiekszego sredniego outstanding
order = wide.mean().sort_values(ascending=False).index.tolist()
wide = wide[order]
piv = wide  # alias dla chart3-plot

print(f"\nEvent dates: {len(piv)}, range: {piv.index.min().date()} → {piv.index.max().date()}")
print(f"Latest total: {piv.iloc[-1].sum() / 1000:.1f} bln PLN")
print(f"Latest per type: {piv.iloc[-1].to_dict()}")
print(f"Types: {order}")
piv.tail()

In [ ]:
fig = go.Figure()
for bt in order:
    fig.add_trace(go.Scatter(
        x=piv.index, y=piv[bt] / 1000.0,
        name=bt, mode="lines", stackgroup="one",
        hovertemplate="%{y:.1f} bln PLN<extra>" + bt + "</extra>",
    ))

# Niewidoczna linia na szczycie stack-u zeby unified hover pokazywal Σ TOTAL.
totals_bln = piv.sum(axis=1) / 1000.0
fig.add_trace(go.Scatter(
    x=piv.index, y=totals_bln,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{y:.1f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="Skład długu skarbowego (bondy hurtowe + bony) — auction events",
    xaxis_title="Data aukcji / odkupu",
    yaxis_title="Outstanding (bln PLN)",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 3b. Skład długu — udział procentowy per rodzaj kuponu

Re-bucketing chart 3 z **per bond_type** (DS/WZ/PS/...) na **per coupon kind**:
- **I** — inflacyjne (IZ)
- **OS** — stałe + zerokuponowe (OK, DS, PS, WS, OS, PP, AS, CK, KO, DK, DZ, PK, RP, TK)
- **Z** — zmienne (WZ, NZ)
- **tbill** — bony skarbowe (osobny kubełek)

Wykres jako stacked area znormalizowany do 100% — pokazuje jak ewoluowała **struktura procentowa** zadłużenia (nie nominalna).

In [ ]:
# to_bucket() i BOND_TYPE_TO_BUCKET sa zdefiniowane w chart2-data (reuse).
# Reuse piv (wide DF z chart3-data, juz pelne ffilled per-day + zredukowane
# do event-only rows). Sumujemy kolumny tego samego bucketu.
piv_buckets = pd.DataFrame(index=piv.index)
for bt in piv.columns:
    bucket = to_bucket(bt)
    if bucket not in piv_buckets.columns:
        piv_buckets[bucket] = 0.0
    piv_buckets[bucket] = piv_buckets[bucket] + piv[bt]

# Procenty per row (udzial w calosci emisji na ta date)
row_sums = piv_buckets.sum(axis=1)
piv_pct = piv_buckets.div(row_sums.replace(0, pd.NA), axis=0) * 100
piv_pct = piv_pct.fillna(0)

# Order rysowania: OS na dole (najwiekszy), potem Z, I, tbill na gorze
BUCKET_ORDER = ["OS", "Z", "I", "tbill"]
piv_pct = piv_pct[[c for c in BUCKET_ORDER if c in piv_pct.columns]]
piv_buckets = piv_buckets[piv_pct.columns]

print(f"Buckets: {list(piv_pct.columns)}")
print(f"Latest total: {row_sums.iloc[-1] / 1000:.1f} bln PLN")
print(f"Latest % per bucket: "
      f"{ {k: f'{v:.1f}%' for k, v in piv_pct.iloc[-1].to_dict().items()} }")
print(f"Latest nominal per bucket (mln PLN): "
      f"{ {k: f'{v:,.0f}' for k, v in piv_buckets.iloc[-1].to_dict().items()} }")
piv_pct.tail()

In [ ]:
BUCKET_COLORS = {
    "OS": "#1f77b4",      # niebieski - stale+zerokup (dominanta)
    "Z": "#ff7f0e",       # pomaranczowy - zmienne (WZ/NZ)
    "I": "#9467bd",       # fioletowy - inflacyjne (IZ)
    "tbill": "#7f7f7f",   # szary - bony skarbowe
}

fig = go.Figure()
totals_bln_3b = row_sums / 1000.0  # bln PLN per data (do hover-a)

for bucket in piv_pct.columns:
    nominal_bln = piv_buckets[bucket] / 1000.0
    # customdata = nominal w bln, zeby pokazac obok % w hoverze
    fig.add_trace(go.Scatter(
        x=piv_pct.index, y=piv_pct[bucket],
        customdata=nominal_bln,
        name=bucket, mode="lines", stackgroup="one",
        line=dict(width=0.5, color=BUCKET_COLORS.get(bucket, "grey")),
        hovertemplate=("%{y:.1f}%  (%{customdata:.1f} bln PLN)"
                       "<extra>" + bucket + "</extra>"),
    ))

# Σ TOTAL trace - niewidoczna linia na szczycie (y=100 zeby zawsze byla na
# top edge), w hoverze pokazuje absolutny total w bln PLN.
fig.add_trace(go.Scatter(
    x=piv_pct.index, y=[100.0] * len(piv_pct),
    customdata=totals_bln_3b,
    name="Σ TOTAL", mode="lines",
    line=dict(color="rgba(0,0,0,0)", width=0),
    hovertemplate="<b>Σ TOTAL</b>: %{customdata:.1f} bln PLN<extra></extra>",
    showlegend=False,
    hoverlabel=dict(bgcolor="black", font=dict(color="white")),
))

fig.update_layout(
    title="Skład długu skarbowego — udział procentowy per rodzaj kuponu",
    xaxis_title="Data aukcji / odkupu",
    yaxis_title="Udział (%)",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    template="plotly_white",
    hovermode="x unified",
    height=600,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

## 4. Aukcje obligacji skarbowych

Dane z arkusza MF Operacje (per leg). Filtr: `type_tx='S' AND type_op IN ('AS','AU','AZ')` — aukcje sprzedażowe (primary + top-up) **plus sale leg aukcji zamiany (switch)**. Odkupy (AO) i buyback leg AZ pominięte (B/C i concession nie mają tam sensu).

**Note dot. tail-a:** Polskie aukcje hurtowe od ~2010 są **holenderskie (uniform-price)** — wszyscy płacą cenę cut-off, więc tail (max yield - avg yield) = 0 by construction. Nie pokazujemy tego więcej, zostawiamy **concession** (cut-off yield vs same-day fixing morning) jako miarę „drogo czy through".

**AZ specifika:** sale leg aukcji zamiany to sprzedaż nowej serii w wymianie za starą. Bid/cover liczone jak normalnie (demand/sold), concession też (yield_avg vs prior fixing). Wyniki czasem niższe niż AS/AU bo demand jest często „captive" (inwestorzy chcący się zrolować).

Sekcje:
- **4a** — statystyki dla CAŁEJ aukcji dziennej (wszystkie serie razem, AS+AU+AZ-sale)
- **4b** — per typ kuponu (I = inflacja, OS = O+S zerokuponowe i stałe, Z = zmienne)
- **4c** — szczegółowa tabela ostatniej aukcji
- **4d** — box plots: dzisiejsze serie vs ich pełna historia (B/C + concession)

### 4a. Statystyki per aukcja (cały dzień)

Każda aukcja = wszystkie sprzedane tego dnia serie sumarycznie. Górny wykres: B/C, dolny: concession (cut-off vs same-day fixing 1).

In [ ]:
df4a = fetch_view(
    "v_auction_day_totals",
    f"?type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=*&order=auction_date.asc",
)
df4a["auction_date"] = pd.to_datetime(df4a["auction_date"])
for c in ["bid_to_cover", "bid_to_offer", "w_yield_avg",
          "w_concession_bp", "total_sold_mln", "total_demand_mln",
          "nc_share_demand"]:
    if c in df4a.columns:
        df4a[c] = pd.to_numeric(df4a[c], errors="coerce")

agg = df4a.groupby("auction_date", as_index=False).agg(
    total_sold_mln=("total_sold_mln", "sum"),
    total_demand_mln=("total_demand_mln", "sum"),
    w_yield_avg=("w_yield_avg", "mean"),
    w_concession_bp=("w_concession_bp", "mean"),
)
agg["bid_to_cover"] = agg["total_demand_mln"] / agg["total_sold_mln"]

# Rolling mean B/C - 12 aukcji ≈ kwartal (przy ~1 aukcji/tydzien). Window
# wyrownuje pojedyncze outliers (np. illiquid AZ z low demand), pokazuje
# strukturalny trend popytu vs jednoaukcyjne wahania.
MA_WINDOW = 12
agg = agg.sort_values("auction_date").reset_index(drop=True)
agg["bid_to_cover_ma"] = (
    agg["bid_to_cover"].rolling(window=MA_WINDOW, min_periods=4).mean()
)

print(f"Auction days: {len(agg)},  range: {agg.auction_date.min().date()} → {agg.auction_date.max().date()}")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover + total sold", "Concession (bp)"),
                    specs=[[{"secondary_y": True}], [{}]])

fig.add_trace(go.Bar(x=agg["auction_date"], y=agg["total_sold_mln"],
                     name="Total sold (mln PLN)", marker_color="lightgrey",
                     opacity=0.6), row=1, col=1, secondary_y=True)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover"],
                         name="Bid-to-cover", mode="lines+markers",
                         line=dict(color="navy", width=1.5),
                         marker=dict(size=4),
                         opacity=0.55),
              row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["bid_to_cover_ma"],
                         name=f"B/C MA({MA_WINDOW} aukcji)",
                         mode="lines",
                         line=dict(color="darkred", width=2.5)),
              row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(x=agg["auction_date"], y=agg["w_concession_bp"],
                         name="Concession (bp)", mode="lines+markers",
                         line=dict(color="seagreen", width=1.5)), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

fig.update_yaxes(title_text="B/C", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="mln PLN", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="bp", row=2, col=1)

fig.update_layout(
    title="Statystyki polskich aukcji obligacji skarbowych (AS+AU+AZ-sale)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.show()

### 4b. B/C i concession per typ kuponu (I / OS / Z)

- **I** — inflation-linked (Oprocentowanie='I', np. IZ)
- **OS** — zero-coupon + stałe łącznie (O+S, np. OK + PS + DS + WS)
- **Z** — zmienne (WZ, NZ)

In [ ]:
df4b = fetch_view(
    "v_auction_by_coupon_bucket",
    f"?type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=*&order=auction_date.asc,coupon_bucket.asc",
)
df4b["auction_date"] = pd.to_datetime(df4b["auction_date"])
for c in ["bid_to_cover", "w_yield_avg", "w_concession_bp", "total_sold_mln"]:
    df4b[c] = pd.to_numeric(df4b[c], errors="coerce")

print(f"Rows: {len(df4b)},  buckets: {sorted(df4b.coupon_bucket.unique())}")

BUCKET_COLORS = {"I": "darkviolet", "OS": "navy", "Z": "darkorange"}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=("Bid-to-cover", "Concession (bp)"))

for bucket in sorted(df4b.coupon_bucket.unique()):
    sub = df4b[df4b.coupon_bucket == bucket].sort_values("auction_date")
    color = BUCKET_COLORS.get(bucket, "grey")
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["bid_to_cover"],
                             name=bucket, legendgroup=bucket, mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=sub["auction_date"], y=sub["w_concession_bp"],
                             name=bucket, legendgroup=bucket, showlegend=False,
                             mode="lines+markers",
                             line=dict(color=color, width=1.5), marker=dict(size=4)),
                  row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)
fig.update_layout(
    title="Metryki aukcyjne per kubełek kuponu (AS+AU+AZ-sale)",
    template="plotly_white", height=700,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.show()

### 4c. Ostatnia aukcja — szczegóły per seria

Tabela wszystkich serii sprzedanych w najnowszej dacie. `concession_bp` > 0 = aukcja droga vs rynek wtórny, < 0 = "through" (taniej niż rynek).

In [ ]:
df4d_recent = fetch_view(
    "v_recent_auctions",
    "?type_op=in.(AS,AU,AZ)&select=*&order=auction_date.desc&limit=50",
)
df4d_recent["auction_date"] = pd.to_datetime(df4d_recent["auction_date"]).dt.date

latest_date = df4d_recent["auction_date"].max()
df_last = df4d_recent[df4d_recent["auction_date"] == latest_date].copy()
for c in ["offer_max_mln", "demand_total_mln", "sold_total_mln",
          "bid_to_cover", "yield_avg",
          "concession_bp", "nc_share_demand"]:
    if c in df_last.columns:
        df_last[c] = pd.to_numeric(df_last[c], errors="coerce")

total_sold = df_last["sold_total_mln"].sum()
total_demand = df_last["demand_total_mln"].sum()
overall_bc = total_demand / total_sold if total_sold else float("nan")

print(f"Ostatnia aukcja: {latest_date}  ({len(df_last)} serii)")
print(f"Łączny sold:    {total_sold:>10,.1f} mln PLN")
print(f"Łączny demand:  {total_demand:>10,.1f} mln PLN")
print(f"B/C całej aukcji: {overall_bc:.2f}")

cols = ["seria", "type_op", "years_to_maturity", "coupon_kind", "offer_max_mln",
        "demand_total_mln", "sold_total_mln", "bid_to_cover",
        "yield_avg", "concession_bp", "nc_share_demand"]
view = df_last[cols].rename(columns={
    "type_op": "op",
    "years_to_maturity": "yrs",
    "coupon_kind": "kind",
    "offer_max_mln": "offer",
    "demand_total_mln": "demand",
    "sold_total_mln": "sold",
    "bid_to_cover": "B/C",
    "yield_avg": "yld_avg %",
    "concession_bp": "conc bp",
    "nc_share_demand": "NK %",
})
view["NK %"] = view["NK %"] * 100
view.style.format({
    "offer": "{:,.0f}", "demand": "{:,.0f}", "sold": "{:,.0f}",
    "B/C": "{:.2f}", "yld_avg %": "{:.3f}",
    "conc bp": "{:+.1f}", "NK %": "{:.1f}",
}, na_rep="—")

### 4d. Historyczna dystrybucja per seria — box ploty

Dla każdej serii sprzedanej w ostatniej aukcji (4c):
- **Box plot** całej historii (od 2012) tej **konkretnej** serii: wszystkie wcześniejsze aukcje (AS+AU+AZ-sale).
- **Czerwony diament** = wartość z dzisiejszej aukcji — od razu widać czy dzisiaj jest "typowo", ekstremalnie czy outlier.

Pusty box (sama czerwona kropka) = brand new series, brak historii.

Dwie metryki:
- góra: **Bid-to-cover** (referencyjna linia y=1)
- dół: **Concession bp** (referencyjna linia y=0)

In [ ]:
# Wszystkie historyczne sale legs (AS+AU+AZ) od 2012-01-01 z B/C i concession.
# v_recent_auctions juz filtruje type_tx='S' AND type_op IN ('AS','AU','AZ').
all_hist = fetch_view(
    "v_recent_auctions",
    f"?auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,seria,bid_to_cover,concession_bp"
    "&order=auction_date.asc",
)
all_hist["auction_date"] = pd.to_datetime(all_hist["auction_date"]).dt.date
all_hist["bid_to_cover"] = pd.to_numeric(all_hist["bid_to_cover"], errors="coerce")
all_hist["concession_bp"] = pd.to_numeric(all_hist["concession_bp"], errors="coerce")

# Lista serii sprzedanych dzisiaj (z df_last w chart 4c, juz w scope)
today_series_list = sorted(df_last["seria"].dropna().unique().tolist())

# Filter do tych serii, wyklucz dzisiejsza aukcje (zeby box nie zawieral
# punktu ktory rysujemy osobno czerwonym diamentem)
df_hist = all_hist[
    all_hist["seria"].isin(today_series_list)
    & (all_hist["auction_date"] < latest_date)
].copy()

print(f"Today's series ({latest_date}): {today_series_list}")
print(f"Total historical auctions of these series: {len(df_hist)}")
for s in today_series_list:
    n = len(df_hist[df_hist["seria"] == s])
    print(f"  {s}: {n} prior auctions")

In [ ]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
    subplot_titles=("Historyczne Bid-to-cover per seria (od 2012)",
                    "Historyczne Concession bp per seria (od 2012)"),
)

today_marker = dict(
    symbol="diamond", size=14, color="red",
    line=dict(color="black", width=1),
)

for i, s in enumerate(today_series_list):
    hist = df_hist[df_hist.seria == s]
    today_row = df_last[df_last.seria == s].iloc[0]
    today_bc = pd.to_numeric(today_row.get("bid_to_cover"), errors="coerce")
    today_conc = pd.to_numeric(today_row.get("concession_bp"), errors="coerce")
    n_hist = len(hist)

    # Box B/C
    fig.add_trace(go.Box(
        y=hist["bid_to_cover"], name=s,
        marker_color="lightblue", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:.2f}<extra></extra>"
        ),
    ), row=1, col=1)
    # Today's value B/C marker
    if pd.notna(today_bc):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_bc],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=(i == 0),
            hovertemplate=f"<b>{s}</b><br>dzisiaj B/C: %{{y:.2f}}<extra></extra>",
        ), row=1, col=1)

    # Box concession
    fig.add_trace(go.Box(
        y=hist["concession_bp"], name=s,
        marker_color="lightgreen", boxmean=True,
        showlegend=False,
        hovertemplate=(
            f"<b>{s}</b> (n={n_hist})<br>"
            "%{y:+.1f} bp<extra></extra>"
        ),
    ), row=2, col=1)
    # Today's value concession marker
    if pd.notna(today_conc):
        fig.add_trace(go.Scatter(
            x=[s], y=[today_conc],
            mode="markers", marker=today_marker,
            name="dzisiaj", legendgroup="today",
            showlegend=False,
            hovertemplate=f"<b>{s}</b><br>dzisiaj conc: %{{y:+.1f}} bp<extra></extra>",
        ), row=2, col=1)

fig.add_hline(y=1, line_dash="dot", line_color="grey", row=1, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="grey", row=2, col=1)

fig.update_layout(
    title=f"Aukcja {latest_date}: dzisiaj vs pełna historia per seria",
    template="plotly_white",
    height=750,
    showlegend=True,
    legend=dict(orientation="h", y=-0.10),
)
fig.update_yaxes(title_text="B/C", row=1, col=1)
fig.update_yaxes(title_text="bp", row=2, col=1)
fig.update_xaxes(title_text="Seria", row=2, col=1)
fig.show()

## 5. Wpływ aukcji na portfolio metryki

Każda aukcja sprzedażowa (AS/AU) zmienia ważone outstanding metryki portfela: ATM, ATR, Mod/Mac Duration. Wykres pokazuje **pure composition impact** — delta metryki na dzień aukcji vs poprzedni dzień handlowy z odjętą time-decay (~1/365 per dzień kalendarzowy).

Interpretacja:
- **Δ > 0** (niebieski) — aukcja **wydłużyła** średnią charakterystykę portfela (typowo nowa długa seria DS/WS/IZ)
- **Δ < 0** (czerwony) — aukcja **skróciła** (np. OK krótka, albo re-open krótkiej istniejącej serii)
- **Δ ≈ 0** — re-open serii z metryką bliską portfolio average (nominal change tylko wagi)

ATM ma czysty time decay (wszyscy = -1/365 dziennie) więc po korekcie zostaje praktycznie wyłącznie composition impact. Dla Mod/Mac jest to przybliżenie (zakłada flat yield curve day-over-day).

In [ ]:
# Fetch aukcje (sale legs AS/AU/AZ-sale) z listami serii per dzien.
auctions_5 = fetch_view(
    "bond_auctions",
    f"?type_tx=eq.S&type_op=in.(AS,AU,AZ)&auction_date=gte.{START_DATE_STR}"
    "&select=auction_date,seria,sold_total_mln&order=auction_date.asc",
)
auctions_5["auction_date"] = pd.to_datetime(auctions_5["auction_date"])
auctions_5["sold_total_mln"] = pd.to_numeric(auctions_5["sold_total_mln"], errors="coerce")

day_agg_5 = auctions_5.groupby("auction_date").agg(
    total_sold_mln=("sold_total_mln", "sum"),
    series_list=("seria", lambda s: ", ".join(sorted(set(s)))),
).reset_index().set_index("auction_date")

# df1 z chart 1 (portfolio_metrics_daily) - liczymy delta per metryka
# vs poprzedni fixing day + day gap dla time-decay correction.
mts = df1.set_index("fixing_date").sort_index()
PORT_COLS = ["portfolio_mod_duration", "portfolio_mac_duration",
             "portfolio_atm", "portfolio_atr"]
for col in PORT_COLS:
    mts[f"delta_{col}"] = mts[col].diff()
mts["gap_days"] = pd.Series(mts.index, index=mts.index).diff().dt.days

# Join na auction_date. Auction zwykle jest tez fixing day (BondSpot codziennie
# pon-pt) wiec to powinno match-owac dla 99% aukcji.
df5 = day_agg_5.join(mts[[f"delta_{c}" for c in PORT_COLS] + ["gap_days"]], how="inner")

# Pure composition impact = raw delta + time-decay correction (gap_days/365).
# Dla ATM to identycznosc matematyczna (wszyscy bondy traca 1/calendar_year),
# dla Mod/Mac przyblizenie (zakladamy flat YC d-o-d).
for col in PORT_COLS:
    df5[f"impact_{col}"] = df5[f"delta_{col}"] + df5["gap_days"] / 365.0

print(f"Auctions joined with portfolio metrics: {len(df5)} "
      f"(range {df5.index.min().date()} → {df5.index.max().date()})")
print(f"Avg impact: mod={df5['impact_portfolio_mod_duration'].mean():+.4f}Y, "
      f"atm={df5['impact_portfolio_atm'].mean():+.4f}Y")
print(f"Biggest single impact (mod_duration): "
      f"{df5['impact_portfolio_mod_duration'].abs().idxmax().date()} "
      f"({df5['impact_portfolio_mod_duration'].abs().max():.4f}Y)")
df5.tail(10)

In [ ]:
metrics_5 = [
    ("impact_portfolio_mod_duration", "Δ Modified Duration (lata)"),
    ("impact_portfolio_mac_duration", "Δ Macaulay Duration (lata)"),
    ("impact_portfolio_atm",          "Δ ATM (lata)"),
    ("impact_portfolio_atr",          "Δ ATR (lata)"),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics_5],
                    shared_xaxes=True, vertical_spacing=0.10, horizontal_spacing=0.08)

# Niebieski dla +, czerwony dla -, szary dla blisko zera (|Δ| < 1bp ATM)
def _color(v):
    if pd.isna(v):
        return "lightgrey"
    if abs(v) < 0.0001:
        return "lightgrey"
    return "#1f77b4" if v > 0 else "#d62728"

customdata = list(zip(df5["total_sold_mln"], df5["series_list"]))

for i, (col, label) in enumerate(metrics_5):
    row, c = i // 2 + 1, i % 2 + 1
    colors = [_color(v) for v in df5[col]]
    fig.add_trace(
        go.Bar(
            x=df5.index, y=df5[col],
            marker_color=colors,
            name=label, showlegend=False,
            customdata=customdata,
            hovertemplate=(
                "<b>%{x|%Y-%m-%d}</b><br>"
                f"{label}: " + "%{y:+.4f} Y<br>"
                "sold: %{customdata[0]:,.0f} mln PLN<br>"
                "series: %{customdata[1]}<extra></extra>"
            ),
        ),
        row=row, col=c,
    )
    fig.add_hline(y=0, line_dash="dot", line_color="grey", row=row, col=c)

fig.update_layout(
    title="Wpływ poszczególnych aukcji na portfolio metryki "
          "(composition impact, time-decay odjete)",
    template="plotly_white",
    height=750,
    hovermode="closest",
    bargap=0.1,
)
fig.show()

---

**Tip:** wykresy plotly są interaktywne — najedź myszą żeby zobaczyć wartości, zaznacz prostokąt żeby przybliżyć, podwójny klik żeby zresetować. Trace w legendzie można wyłączać klikiem.